# Checking image before download

In [ ]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
from helpers import config
from helpers import utils

In [10]:
config.ROI =  ee.Geometry.Polygon(
        [[[84.04000703052756, 27.781872772191807],
          [84.04000703052756, 27.77731646544985],
          [84.04704514698264, 27.77731646544985],
          [84.04704514698264, 27.781872772191807]]], None, False);

seg_bands = ee.List(["blue", "green", "red", "nir", "swir1", "swir2", "ndvi", "ndmi", "ndvi_shade", "ndvi_savg"])
indices_formula = {
    'NDVI': 'clamp((b("nir") - b("red")) / (b("nir") + b("red")), -1, 1)',
    'EVI': (
        'clamp(2.5 * ((b("nir") - b("red")) / '
        '     clamp(b("nir") + 6 * b("red") - 7.5 * b("blue") + 1, 0, 10)), '
        '   -1.0, 1.0)'
    ),
    'NBR': 'clamp((b("nir") - b("swir2")) / (b("nir") + b("swir2")), -1, 1)',
    'NDWI': 'clamp((b("green") - b("nir")) / (b("green") + b("nir")), -1, 1)',
    'NDMI': 'clamp((b("nir") - b("swir1")) / (b("nir") + b("swir1")), -1, 1)',
    'BAI': '1.0 / (pow(0.1 - b("red"), 2) + pow(0.06 - b("nir"), 2))',
    'MSAVI': '(2 * b("nir") + 1 - sqrt(pow(2 * b("nir") + 1, 2) - 8 * (b("nir") - b("red")))) / 2.0',
}

def add_indices(image):
    ndvi = image.expression(indices_formula["NDVI"]).rename("ndvi")
    ndmi = image.expression(indices_formula["NDMI"]).rename("ndmi")
    evi = image.expression(indices_formula["EVI"]).rename("evi")
    msavi = image.expression(indices_formula["MSAVI"]).rename("msavi")
    return image.addBands(ndvi).addBands(ndmi).addBands(evi).addBands(msavi)

print("Building seg_image_stack...")
l_winter2002_image = utils.get_processed_landsat_collection("LANDSAT/LE07/C02/T1_L2", config.ROI, ["2002-12-01", "2003-04-01"], utils.mask_clouds_landsat75, utils.apply_scale_factors).median().select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)
l_winter2002_image = add_indices(l_winter2002_image)
high_val_winter2002 = ee.Number(utils.calc_image_stats(l_winter2002_image, config.ROI, config.SCALE).get("high"))
l_winter2002_image = utils.add_scaled_glcm(l_winter2002_image, config.ROI, config.SCALE, high_val_winter2002)#.select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2002")))

l_winter2010_image = utils.get_processed_landsat_collection("LANDSAT/LT05/C02/T1_L2", config.ROI, ["2010-12-01", "2011-03-01"],utils.mask_clouds_landsat75, utils.apply_scale_factors).median().select(config.L75_ORIGINAL_BAND_NAMES).rename(config.L75_NEW_BAND_NAMES)
l_winter2010_image = add_indices(l_winter2010_image)
high_val_winter2010 = ee.Number(utils.calc_image_stats(l_winter2010_image, config.ROI, config.SCALE).get("high"))
l_winter2010_image = utils.add_scaled_glcm(l_winter2010_image, config.ROI, config.SCALE, high_val_winter2010).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2010")))

l_preMons_image = utils.get_processed_landsat_collection("LANDSAT/LC08/C02/T1_L2", config.ROI, ["2025-12-01", "2026-03-01"], utils.mask_clouds_landsat8, utils.apply_scale_factors).median().select(config.L8_ORIGINAL_BAND_NAMES).rename(config.L8_NEW_BAND_NAMES)
l_preMons_image = add_indices(l_preMons_image)
high_val_preMons = ee.Number(utils.calc_image_stats(l_preMons_image, config.ROI, config.SCALE).get("high"))
l_preMons_image = utils.add_scaled_glcm(l_preMons_image, config.ROI, config.SCALE, high_val_preMons).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2017")))

seg_image_stack = l_winter2002_image.addBands(l_winter2010_image).addBands(l_preMons_image)

Building seg_image_stack...


In [11]:
seg_image_stack

In [12]:
vis_params = {"min":0, "max":0.3}
Map.addLayer(seg_image_stack, vis_params, "seg_stack")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

EEException: Reprojection output too large (11972x5790 pixels).

# Downloading Images
## Note that downloading Landsat and GLCM image separately to avoid memory errors

In [1]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-municipalites-climate")
import geemap
Map = geemap.Map()
from helpers import landsat_composites
from helpers import config
from helpers import utils
print(ee.data._get_state().cloud_api_user_project)

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


ee-joshisur231


In [2]:
# config.ROI =     ee.Geometry.Polygon(
#         [[[82.7265858332212, 28.339059624310533],
#           [82.7265858332212, 28.303699048119494],
#           [82.77877089181494, 28.303699048119494],
#           [82.77877089181494, 28.339059624310533]]], None, False);
scale = 30
seg_bands = ee.List(['green', 'red', 'nir', 'swir1', 'swir2', 'ndvi'])
# seg_bands = ee.List(['blue','green','red','nir','swir1','swir2','ndvi','evi','ndmi','msavi', 'ndvi_shade','ndvi_savg'])
processor = landsat_composites.C2SRExpressions()
def compute_indices(image):
    ndvi = image.expression(processor.ALGORITHMS["NDVI"]).rename("ndvi")
    # evi = image.expression(processor.ALGORITHMS["EVI"]).rename("evi")
    ndmi = image.expression(processor.ALGORITHMS["NDMI"]).rename("ndmi")
    # msavi = image.expression(processor.ALGORITHMS["MSAVI"]).rename("msavi")
    return image.addBands([ndvi]) #([ndvi, evi, ndmi, msavi])


unified_collection = processor._expression()\
    .map(compute_indices)\
    .select(['green', 'red', 'nir', 'swir1', 'swir2', 'ndvi'])
    #.select(['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'ndvi', 'evi', 'ndmi', 'msavi'])

In [3]:
dem = ee.Image("USGS/SRTMGL1_003").rename("z")
terrain = ee.Terrain.products(dem)
l_winter2002_image1 = unified_collection.filterBounds(config.ROI).filterDate("1999-01-01", "2002-01-01").filter(ee.Filter.calendarRange(12, 12, 'month'))
l_winter2002_image2 = unified_collection.filterBounds(config.ROI).filterDate("1999-01-01", "2002-01-01").filter(ee.Filter.calendarRange(1, 3, 'month'))
l_winter2002_image = l_winter2002_image1.merge(l_winter2002_image2).median().rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2000")))
# high_val_winter2002 = ee.Number(utils.calc_image_stats(l_winter2002_image, config.ROI, 100).get("high"))
# l_winter2002_image = utils.add_scaled_glcm(l_winter2002_image, config.ROI, config.SCALE, high_val_winter2002).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2000")))

# terrain = utils.prepare_terrain_seg(terrain, config.ROI, scale = config.SCALE, high_val=high_val_winter2002)

l_winter2010_image1 = unified_collection.filterBounds(config.ROI).filterDate("2008-01-01", "2011-01-01").filter(ee.Filter.calendarRange(12, 12, 'month'))
l_winter2010_image2 = unified_collection.filterBounds(config.ROI).filterDate("2008-01-01", "2011-01-01").filter(ee.Filter.calendarRange(1, 3, 'month'))
l_winter2010_image = l_winter2010_image1.merge(l_winter2010_image2).median().rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2009")))
# high_val_winter2010 = ee.Number(utils.calc_image_stats(l_winter2010_image, config.ROI, config.SCALE).get("high"))
# l_winter2010_image = utils.add_scaled_glcm(l_winter2010_image, config.ROI, config.SCALE, high_val_winter2010).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2010")))

l_winter2024_image1 = unified_collection.filterBounds(config.ROI).filterDate("2023-01-01", "2026-01-01").filter(ee.Filter.calendarRange(12, 12, 'month'))
l_winter2024_image2 = unified_collection.filterBounds(config.ROI).filterDate("2023-01-01", "2026-01-01").filter(ee.Filter.calendarRange(1, 3, 'month'))
l_winter2024_image = l_winter2024_image1.merge(l_winter2024_image2).median().rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2024")))
# l_preMons_image = unified_collection.filterBounds(config.ROI).filterDate("2025-12-01", "2026-03-01").median()
# high_val_preMons = ee.Number(utils.calc_image_stats(l_preMons_image, config.ROI, config.SCALE).get("high"))
# l_preMons_image = utils.add_scaled_glcm(l_preMons_image, config.ROI, config.SCALE, high_val_preMons).select(seg_bands).rename(seg_bands.map(lambda band_name: ee.String(band_name).cat("_2024")))

seg_image_stack = l_winter2002_image.addBands(l_winter2010_image).addBands(l_winter2024_image)#.addBands(terrain).addBands(dem.rename("z"))

In [34]:
geemap.ee_export_image_to_asset(
    image=l_winter2002_image.clip(config.ROI),
    description=f'l_2000',
    assetId=f'projects/ee-municipalites-climate/assets/image_segmentation_stack/winter2000_image',
    region=config.ROI,
    scale=scale,
    crs='EPSG:32645',
    maxPixels=1e13
)
geemap.ee_export_image_to_asset(
    image=l_winter2010_image.clip(config.ROI),
    description=f'l_2009',
    assetId=f'projects/ee-municipalites-climate/assets/image_segmentation_stack/winter2009_image',
    region=config.ROI,
    scale=scale,
    crs='EPSG:32645',
    maxPixels=1e13
)
geemap.ee_export_image_to_asset(
    image=l_winter2024_image.clip(config.ROI),
    description=f'l_2024',
    assetId=f'projects/ee-municipalites-climate/assets/image_segmentation_stack/winter2024_image',
    region=config.ROI,
    scale=scale,
    crs='EPSG:32645',
    maxPixels=1e13
)
# geemap.ee_export_image_to_asset(
#     image=terrain.clip(config.ROI),
#     description=f'terrain',
#     assetId=f'projects/ee-municipalites-climate/assets/image_segmentation_stack/terrain_image',
#     region=config.ROI,
#     scale=scale,
#     crs='EPSG:32645',
#     maxPixels=1e13
# )

In [6]:
vis_params = {"min":0, "max":0.3, "bands": ["red", "green", "blue"]}
Map.addLayer(l_winter2010_image.clip(config.ROI), vis_params, "2002")
# Map.addLayer(l_winter2010_image.clip(config.ROI), vis_params, "2010")
# Map.addLayer(l_preMons_image.clip(config.ROI), vis_params, "2025")
Map

Map(bottom=27914.0, center=[27.225325836903373, 87.09820857606479], controls=(WidgetControl(options=['position…

# Downloading GLCM Images

In [1]:
import ee
from gee_init import list_accounts, initialize_gee
initialize_gee(account="work1", project="ee-municipalites-climate")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


'account:work1:ee-municipalites-climate'

In [2]:
import geemap
Map = geemap.Map()

In [3]:
scale = 30
crs = 'EPSG:32645'
roi  = ee.FeatureCollection("projects/ee-joshisur231/assets/pa_effectiveness/nepal_boundary").first().geometry()
landsat_image_collection = ee.ImageCollection("projects/ee-municipalites-climate/assets/image_segmentation_stack")
winter2000 = ee.Image('projects/ee-municipalites-climate/assets/image_segmentation_stack/winter2000_image')
winter2009= ee.Image('projects/ee-municipalites-climate/assets/image_segmentation_stack/winter2009_image')
winter2024= ee.Image('projects/ee-municipalites-climate/assets/image_segmentation_stack/winter2024_image')

### GLCM Computation

In [4]:
#1 Get the NDVI image stat to stretch at specified values
def get_image_stats(image, scale, roi):
    stats = image.reduceRegion(
        reducer=ee.Reducer.percentile([2, 98]),
        geometry=roi, 
        scale=scale, 
        maxPixels=1e13,
        tileScale = 16
    )
    return stats

#2 Compute GLCM 
def compute_glcm(image, glcm_band, min_val, max_val):
    image = image.select(glcm_band).updateMask(image.select(glcm_band).gte(0))
    rescaled_image = image.unitScale(min_val, max_val).clamp(0, 1).multiply(100).int()
    glcm_image = rescaled_image.glcmTexture(size=1, average=True)
    
    # Derive band names from the actual input band, not a hardcoded list
    suffixes = ["contrast", "idm", "var", "corr", "savg", "asm"]
    bands_to_select = ee.List(suffixes).map(
        lambda s: ee.String(glcm_band).cat("_").cat(s)
    )
    return glcm_image.select(bands_to_select)

#3 Get landsat image stats to stretch the GLCM image appropriately
# The SNIC will assign higher weight to higher values therefore
# this scaling is important so that GLCM image is not assigned
# very high weight compared to the landsat bands
def calc_image_stats(image, roi, scale):
    stats = image.reduceRegion(
        geometry=roi, 
        maxPixels = 1e13,
        scale=scale, 
        tileScale = 16,
        reducer=ee.Reducer.mean().combine(
            reducer2=ee.Reducer.stdDev(), 
            sharedInputs=True
        )
    )
    def add_3sd_bands(current_band, accum_dict):
        band = ee.String(current_band)
        dict_obj = ee.Dictionary(accum_dict)
        
        mean = ee.Number(dict_obj.get(band.cat("_mean")))
        std = ee.Number(dict_obj.get(band.cat("_stdDev")))
        
        plus3sd = mean.add(std.multiply(3))
        
        return dict_obj \
            .set(band.cat("_plus3sd"), plus3sd)
    stats_final = ee.Dictionary(image.bandNames().iterate(add_3sd_bands, stats))
    
    image_bands = image.bandNames()
    plus3sd_keys = image_bands.map(lambda b: ee.String(b).cat('_plus3sd'))
    plus3sd_values = stats_final.select(plus3sd_keys).values()
    high = ee.Number(plus3sd_values.reduce(ee.Reducer.mean()))
    return high

def get_glcm_min_max(band, image, roi, scale, tileScale=16):
    image = ee.Image(image)
    image = image.select([band])
    minmax_dict = ee.Dictionary()
    image_minMax = image.reduceRegion(reducer=ee.Reducer.minMax(), scale=scale, geometry=roi, maxPixels=1e13, tileScale=tileScale)
    band_min = ee.Number(image_minMax.get(ee.String(band).cat("_min")))
    band_max = ee.Number(image_minMax.get(ee.String(band).cat("_max")))
    minmax_dict = minmax_dict.set('band', band).set('min', band_min).set('max', band_max)
    
    return minmax_dict
    
#4 Rescale the glcm using the values from 3
def rescale_glcm_landsat(image, high_landsat_value, glcm_min, glcm_max, scale, roi, bands, tileScale=16):
    
    def rescale(band, image):
        image = ee.Image(image)
        image = image.select([band])
        # image_minMax = image.reduceRegion(reducer=ee.Reducer.minMax(), scale=scale, geometry=roi, maxPixels=1e13, tileScale=tileScale)
        band_min = ee.Number(glcm_min.get(ee.String(band).cat("_min")))
        band_max = ee.Number(glcm_max.get(ee.String(band).cat("_max")))
        
        final_image = image.unitScale(band_min, band_max).multiply(high_landsat_value)
        return final_image
    final_image = ee.ImageCollection(bands.map(lambda band: rescale(band, image))).toBands().rename(bands)
    
    return final_image

def runall(image, suffix, compute=None):
    if compute == "glcm_source_stats":
        stats = get_image_stats(image.select(ee.String("ndvi").cat("_").cat(suffix)), 30, roi) #only need stats for NDVI
        return stats
    elif compute == "glcm":
        stats = get_image_stats(image.select(ee.String("ndvi").cat("_").cat(suffix)), 30, roi) 
        glcm = compute_glcm(image, ee.String("ndvi").cat("_").cat(suffix), ee.Number(stats.get(ee.String("ndvi").cat("_").cat(suffix).cat(ee.String("_p2")))), ee.Number(stats.get(ee.String("ndvi").cat("_").cat(suffix).cat(ee.String("_p98")))))
        return glcm
    elif compute == "l_high_val":
        l_high_val = calc_image_stats(image.select([ee.String('green').cat("_").cat(suffix), ee.String('red').cat("_").cat(suffix), ee.String('nir').cat("_").cat(suffix), ee.String('swir1').cat("_").cat(suffix), ee.String('swir2').cat("_").cat(suffix)]), roi, 30) #not useing NDVI here
        return l_high_val
    else:
        final_glcm = rescale_glcm_landsat(glcm, l_high_val, 30, roi, glcm.bandNames())
        return final_glcm

In [7]:
#For 2000
# source_stats_2000 = runall(winter2000, ee.String("2000"), compute="glcm_source_stats") #1 calculate NDVI stats for rescaling
glcm_2000 = runall(winter2000, ee.String("2000"), compute="glcm") #1 calc NDVI stats, rescale ndvi and compute glcm
high_val_l_2000 = runall(winter2000, ee.String("2000"), compute="l_high_val") #2 get landsat max to rescale glcm
min_max = glcm_2000.reduceRegion(                                 #2 get glcm min max to rescale glcm
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=scale,
    maxPixels=1e13,
    tileScale=16,
)
geemap.ee_export_vector_to_drive(
    collection   = ee.FeatureCollection([ee.Feature(None, min_max)]),
    description = "glcm_minmax_2000",
    fileFormat = "csv"
    # assetId     = "projects/ee-municipalites-climate/assets/glcm_minMax_values/glcm_minmax_2000",
)
# min_max_2000 = ee.FeatureCollection("projects/ee-municipalites-climate/assets/glcm_minMax_values/glcm_minmax_2000")
# rescaled_glcm2000 = rescale_glcm_landsat(glcm_2000, high_val_l_2000, min_max_2000, min_max_2000, scale, roi, glcm_2000.bands())
# for band_name in ['ndvi_2000_contrast', 'ndvi_2000_idm', 'ndvi_2000_var', 'ndvi_2000_corr', 'ndvi_2000_savg', 'ndvi_2000_asm']:
#     print(band_name)
#     band_minMax = get_glcm_min_max(
#         image = glcm_2000,
#         band="ndvi_median", 
#         roi=roi, 
#         scale=scale, 
#         tileScale=16).getInfo()
#     # glcm_min_max_2000 = ee.Dictionary(
#     #     ee.Dictionary.fromLists(
#     #         glcm_min_max_2000.map(lambda d: ee.Dictionary(d).get("band")),
#     #         glcm_min_max_2000.map(lambda d: ee.Dictionary(d).select(["min", "max"])),
#     #     )
#     # ).getInfo()
#     print(band_minMax)
#     break

Exporting glcm_minmax_2000... Please check the Task Manager from the JavaScript Code Editor.


In [25]:
glcm_2002_source_stats = runall(winter2000, ee.String('2000'), "glcm_source_stats")
glcm_2009_source_stats = runall(winter2009, ee.String('2009'), "glcm_source_stats")
glcm_2024_source_stats = runall(winter2024, ee.String('2024'), "glcm_source_stats")

In [ ]:
geemap.ee_export_image_to_asset(
    image=glcm_2002.clip(roi),
    description=f'glcm_2000',
    assetId=f'projects/ee-municipalites-climate/assets/image_segmentation_stack/glcm2000_image',
    region=roi,
    scale=scale,
    crs=crs,
    maxPixels=1e13,
)
geemap.ee_export_image_to_asset(
    image=glcm_2009.clip(roi),
    description=f'glcm_2009',
    assetId=f'projects/ee-municipalites-climate/assets/image_segmentation_stack/glcm2009_image',
    region=roi,
    scale=scale,
    crs=crs,
    maxPixels=1e13,
)
geemap.ee_export_image_to_asset(
    image=glcm_2024.clip(roi),
    description=f'glcm_2024',
    assetId=f'projects/ee-municipalites-climate/assets/image_segmentation_stack/glcm2024_image',
    region=roi,
    scale=scale,
    crs=crs,
    maxPixels=1e13,
)

# Test above code

In [66]:
test_roi =  ee.Geometry.Polygon(
        [[[82.72761580148291, 28.337397638474272],
          [82.7265858332212, 28.308535376602357],
          [82.75508162179541, 28.30777971477032],
          [82.75937315621924, 28.336793273542266]]], None, False)

test_sr = winter2024
test_ndvi = winter2024
test_stats = get_image_stats(test_ndvi, 30, test_roi)
test_glcm = compute_glcm(test_ndvi, "ndvi_2024", ee.Number(test_stats.get("ndvi_2024_p2")), ee.Number(test_stats.get("ndvi_2024_p98")))
# test_l_high_val = calc_image_stats(test_sr, test_roi, 30)
final_glcm = get_glcm_min_max(ee.String(test_glcm.select(0).bandNames().get(0)), test_glcm.select(0), tileScale=None)

In [67]:
final_glcm

In [6]:
test_glcm.reduceRegion(reducer=ee.Reducer.minMax(), scale=30, geometry=test_roi)

In [7]:
final_glcm.reduceRegion(reducer=ee.Reducer.minMax(), scale=30, geometry=test_roi)

# Principle Component

In [ ]:
# Function to apply PCA to a multi-band image
def get_principal_components(image, region, scale, suffix):
    # Extract the dynamic band names from the input image
    band_names = image.bandNames()
    
    # Mean center the data (Optimized with tileScale and maxPixels for large ROIs)
    mean_dict = image.reduceRegion(
        reducer=ee.Reducer.mean(), 
        geometry=region, 
        scale=scale, 
        maxPixels=1e13,
        tileScale=16
    )
    means = mean_dict.toImage(band_names)
    centered = image.subtract(means)
    
    # Helper to dynamically name output PC bands using the year suffix (e.g., pc1_2024)
    def get_new_band_names(prefix):
        seq = ee.List.sequence(1, band_names.length())
        return seq.map(lambda b: ee.String(prefix).cat(ee.Number(b).int()).cat('_').cat(suffix))

    # Collapse bands into 1D array
    arrays = centered.toArray()

    # Compute the covariance of the bands within the region
    covar = arrays.reduceRegion(
        reducer=ee.Reducer.centeredCovariance(),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        tileScale=16
    )

    covar_array = ee.Array(covar.get('array'))

    # Perform an eigen analysis and slice apart the values and vectors
    eigens = covar_array.eigen()
    eigen_values = eigens.slice(1, 0, 1)
    eigen_vectors = eigens.slice(1, 1)

    # Convert the array image to 2D arrays for matrix computations
    array_image = arrays.toArray(1)

    # Left multiply the image array by the matrix of eigenvectors
    principal_components = ee.Image(eigen_vectors).matrixMultiply(array_image)

    # Turn the square roots of the Eigenvalues into a P-band image
    sd_image = (
        ee.Image(eigen_values.sqrt())
        .arrayProject([0])
        .arrayFlatten([get_new_band_names('sd')])
    )

    # Turn the PCs into a P-band image, normalized by SD
    pc_image = (
        principal_components.arrayProject([0])
        .arrayFlatten([get_new_band_names('pc')])
        .divide(sd_image)
    )
    
    return pc_image

# Apply PCA to your generated GLCM images
pca_2000 = get_principal_components(glcm_2002, roi, 30, ee.String('2000'))
pca_2009 = get_principal_components(glcm_2009, roi, 30, ee.String('2009'))
pca_2024 = get_principal_components(glcm_2024, roi, 30, ee.String('2024'))

# Note: In most image segmentation workflows, the first Principal Component (PC1)
# captures the vast majority of the texture variance. You can isolate PC1 like this:
pc1_2000 = pca_2000.select('pc1_2000')
pc1_2009 = pca_2009.select('pc1_2009')
pc1_2024 = pca_2024.select('pc1_2024')